# Project 1: E-commerce Sales EDA


# Exploratory Data Analysis (EDA)
This notebook follows a reusable EDA workflow: inspect the data, check data quality, summarize distributions, visualize patterns, investigate anomalies, and document findings.

> The dataset is generated inside this notebook so the project runs without downloading files. Replace the sample data with a real CSV later if desired.

## 1. Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option("display.max_columns", 50)
sns.set_theme()

## 2. Create a sample e-commerce dataset

In [ ]:
# Reproducible synthetic data
rng = np.random.default_rng(42)
n = 500
dates = pd.date_range("2025-01-01", periods=180, freq="D")
categories = ["Electronics", "Home", "Clothing", "Books", "Beauty"]
products = {
    "Electronics": ["Headphones", "Keyboard", "Mouse", "Speaker"],
    "Home": ["Lamp", "Bottle", "Pillow", "Storage Box"],
    "Clothing": ["T-shirt", "Jeans", "Jacket", "Socks"],
    "Books": ["Novel", "Notebook", "Cookbook", "Planner"],
    "Beauty": ["Cleanser", "Moisturizer", "Shampoo", "Sunscreen"],
}
category = rng.choice(categories, n)
product = [rng.choice(products[c]) for c in category]
price = rng.uniform(8, 250, n).round(2)
quantity = rng.integers(1, 6, n)
discount = rng.choice([0, 0.05, 0.10, 0.15, 0.20], n, p=[.35,.15,.25,.15,.10])
df = pd.DataFrame({
    "order_id": [f"ORD{i:04d}" for i in range(1, n+1)],
    "order_date": rng.choice(dates, n),
    "category": category,
    "product": product,
    "unit_price": price,
    "quantity": quantity,
    "discount_rate": discount,
    "region": rng.choice(["North", "South", "East", "West"], n),
    "customer_id": rng.integers(1001, 1151, n)
})
df["revenue"] = (df["unit_price"] * df["quantity"] * (1 - df["discount_rate"])).round(2)

# Add a few realistic quality issues for investigation
df.loc[[7, 28, 91], "unit_price"] = np.nan
df.loc[[12, 65], "category"] = np.nan
df = pd.concat([df, df.iloc[[10]]], ignore_index=True)  # intentional duplicate row

df.head()

## 3. Understand the data

In [ ]:
print("Shape:", df.shape)
display(df.head())
display(df.sample(5, random_state=1))
df.info()
display(df.describe(include="all").T)

## 4. Data quality checks

In [ ]:
print("Duplicate rows:", df.duplicated().sum())
missing = df.isna().sum().sort_values(ascending=False)
missing_pct = (df.isna().mean() * 100).round(2)
display(pd.DataFrame({"missing_count": missing, "missing_percent": missing_pct}).query("missing_count > 0"))

# Remove exact duplicate rows and make a working copy
eda = df.drop_duplicates().copy()
# Fill missing category with an explicit label; use median price for missing numeric values
eda["category"] = eda["category"].fillna("Unknown")
eda["unit_price"] = eda["unit_price"].fillna(eda["unit_price"].median())
eda["revenue"] = (eda["unit_price"] * eda["quantity"] * (1 - eda["discount_rate"])).round(2)
print("Shape after duplicate removal:", eda.shape)
print("Remaining missing values:", int(eda.isna().sum().sum()))

## 5. Descriptive statistics and distributions

In [ ]:
numeric_cols = ["unit_price", "quantity", "discount_rate", "revenue"]
display(eda[numeric_cols].describe().T)
print("Revenue variance:", round(eda["revenue"].var(), 2))
print("Revenue standard deviation:", round(eda["revenue"].std(), 2))
print("Revenue skewness:", round(eda["revenue"].skew(), 3))
print("Revenue kurtosis:", round(eda["revenue"].kurtosis(), 3))
display(eda[numeric_cols].quantile([.25, .50, .75]).T.rename(columns={.25:"Q1", .5:"Median", .75:"Q3"}))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(eda["revenue"], kde=True, ax=axes[0])
axes[0].set_title("Order Revenue Distribution")
sns.boxplot(x=eda["revenue"], ax=axes[1])
axes[1].set_title("Order Revenue Box Plot")
plt.tight_layout()
plt.show()

## 6. Category, region, and time patterns

In [ ]:
category_sales = eda.groupby("category", as_index=False).agg(
    orders=("order_id", "count"),
    total_revenue=("revenue", "sum"),
    average_order_revenue=("revenue", "mean")
).sort_values("total_revenue", ascending=False)
display(category_sales)

region_sales = eda.groupby("region", as_index=False)["revenue"].sum().sort_values("revenue", ascending=False)
display(region_sales)

daily_sales = eda.groupby("order_date")["revenue"].sum().sort_index()
monthly_sales = daily_sales.resample("MS").sum()
display(monthly_sales.rename("monthly_revenue").to_frame())

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.barplot(data=category_sales, x="category", y="total_revenue", ax=axes[0])
axes[0].tick_params(axis="x", rotation=30)
axes[0].set_title("Revenue by Category")
monthly_sales.plot(ax=axes[1], marker="o")
axes[1].set_title("Monthly Revenue")
axes[1].set_ylabel("Revenue")
plt.tight_layout()
plt.show()

## 7. Correlation analysis

In [ ]:
corr = eda[["unit_price", "quantity", "discount_rate", "revenue"]].corr(numeric_only=True)
display(corr)
sns.heatmap(corr, annot=True, cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Numeric Feature Correlation")
plt.show()

## 8. Outlier investigation using IQR

In [ ]:
q1 = eda["revenue"].quantile(.25)
q3 = eda["revenue"].quantile(.75)
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
outliers = eda.loc[(eda["revenue"] < lower) | (eda["revenue"] > upper), ["order_id", "order_date", "category", "unit_price", "quantity", "revenue"]]
print(f"IQR bounds: {lower:.2f} to {upper:.2f}")
print("Potential outliers:", len(outliers))
display(outliers.head(10))
# An outlier is a review flag, not automatically an error.

## 9. Optional statistical test

In [ ]:
from scipy.stats import ttest_ind

north = eda.loc[eda["region"] == "North", "revenue"]
south = eda.loc[eda["region"] == "South", "revenue"]
if len(north) > 1 and len(south) > 1:
    statistic, p_value = ttest_ind(north, south, equal_var=False, nan_policy="omit")
    print(f"Welch t-test North vs South: statistic={statistic:.3f}, p-value={p_value:.4f}")
    print("At alpha=0.05:", "evidence of a difference in mean revenue" if p_value < .05 else "insufficient evidence of a difference in mean revenue")
    print("This test does not establish why any difference exists.")

## 10. Key findings template

In [ ]:
print("EDA FINDINGS")
print("- Dataset rows/columns after deduplication:", eda.shape)
print("- Total revenue:", round(eda["revenue"].sum(), 2))
print("- Average order revenue:", round(eda["revenue"].mean(), 2))
print("- Top revenue category:", category_sales.iloc[0]["category"])
print("- Region with highest revenue:", region_sales.iloc[0]["region"])
print("- Missing values after cleaning:", int(eda.isna().sum().sum()))
print("\nReview the charts and outlier table before writing business conclusions.")

## 11. Save cleaned data and report outputs

In [ ]:
output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)
eda.to_csv(output_dir / "ecommerce_cleaned.csv", index=False)
category_sales.to_csv(output_dir / "revenue_by_category.csv", index=False)
plt.figure(figsize=(8, 4))
monthly_sales.plot(marker="o", title="Monthly Revenue")
plt.ylabel("Revenue")
plt.tight_layout()
plt.savefig(output_dir / "monthly_revenue.png", dpi=150)
plt.close()
print("Saved outputs to:", output_dir.resolve())

## Conclusion

Summarize: data quality issues found, cleaning decisions, top category/region, revenue distribution, potential outliers, and any limitations. The sample data is synthetic, so findings are for practice only.